In [1]:
import cv2
import numpy as np

In [ ]:
def get_foreground_mask(diff, threshold=30):
    gray_diff = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY) if len(diff.shape) == 3 else diff
    _, mask = cv2.threshold(gray_diff, threshold, 255, cv2.THRESH_BINARY)
    # Làm sạch noise
    mask = cv2.dilate(mask, None, iterations=2)
    mask = cv2.erode(mask, None, iterations=2)
    return mask


def detect_objects(fgMask, min_area=500):
    contours, _ = cv2.findContours(fgMask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    locations = []
    for contour in contours:
        if cv2.contourArea(contour) < min_area:
            continue
        M = cv2.moments(contour)
        if M['m00'] != 0:
            cx = int(M['m10'] / M['m00'])
            cy = int(M['m01'] / M['m00'])
            locations.append([cx, cy])
    return locations


def draw_detections(frame, fgMask, min_area=500):
    contours, _ = cv2.findContours(fgMask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    result = frame.copy()
    for contour in contours:
        if cv2.contourArea(contour) < min_area:
            continue
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(result, (x, y), (x + w, y + h), (0, 255, 0), 2)
    return result


print("Helper functions defined.")

Helper functions defined.


## 1. Mean Window


In [ ]:
VIDEO_PATH = 'dataset.mp4'
N_WINDOW = 20      # số frame dùng để tính trung bình
TEST_FRAME = 50    # xem kết quả tại frame thứ bao nhiêu

capture = cv2.VideoCapture(VIDEO_PATH)

frame_buffer = []   # lưu n frame gần nhất
frame_idx = 0
result_frame = None
result_mask = None

while True:
    ret, frame = capture.read()
    if not ret:
        break

    frame_float = frame.astype(np.float32)

    if len(frame_buffer) == N_WINDOW:
        # --- Tính background = trung bình n frame trong buffer ---
        background = np.mean(frame_buffer, axis=0).astype(np.uint8)

        # --- Tính foreground mask ---
        diff = cv2.absdiff(frame, background)
        fgMask = get_foreground_mask(diff, threshold=30)

        if frame_idx == TEST_FRAME:
            result_frame = frame.copy()
            result_mask  = fgMask.copy()
            break

        # Trượt cửa sổ: bỏ frame cũ nhất, thêm frame mới
        frame_buffer.pop(0)

    frame_buffer.append(frame_float)
    frame_idx += 1

capture.release()

if result_frame is not None:
    result_display = draw_detections(result_frame, result_mask)
    locations = detect_objects(result_mask)
    print(f"[Mean Window] Frame {TEST_FRAME}:")
    print(f"  Số object phát hiện: {len(locations)}")
    print(f"  Toạ độ tâm object: {locations}")
    print(f"  Sum of intensities in fgMask: {np.sum(result_mask)}")
    cv2.imshow('Mean Window - Frame', result_display)
    cv2.imshow('Mean Window - FG Mask', result_mask)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    print("Không đủ frame để xử lý. Hãy giảm TEST_FRAME hoặc N_WINDOW.")

[Mean Window] Frame 50:
  Số object phát hiện: 3
  Toạ độ tâm object: [[25, 382], [456, 30], [416, 44]]
  Sum of intensities in fgMask: 1721250


## 2. Median Window



In [ ]:
VIDEO_PATH = 'dataset.mp4'
N_WINDOW = 20
TEST_FRAME = 50

capture = cv2.VideoCapture(VIDEO_PATH)

frame_buffer = []
frame_idx = 0
result_frame = None
result_mask = None

while True:
    ret, frame = capture.read()
    if not ret:
        break

    frame_float = frame.astype(np.float32)

    if len(frame_buffer) == N_WINDOW:
        # --- Tính background = trung vị n frame trong buffer ---
        background = np.median(frame_buffer, axis=0).astype(np.uint8)

        diff = cv2.absdiff(frame, background)
        fgMask = get_foreground_mask(diff, threshold=30)

        if frame_idx == TEST_FRAME:
            result_frame = frame.copy()
            result_mask  = fgMask.copy()
            break

        frame_buffer.pop(0)

    frame_buffer.append(frame_float)
    frame_idx += 1

capture.release()

if result_frame is not None:
    result_display = draw_detections(result_frame, result_mask)
    locations = detect_objects(result_mask)
    print(f"[Median Window] Frame {TEST_FRAME}:")
    print(f"  Số object phát hiện: {len(locations)}")
    print(f"  Toạ độ tâm object: {locations}")
    print(f"  Sum of intensities in fgMask: {np.sum(result_mask)}")
    cv2.imshow('Median Window - Frame', result_display)
    cv2.imshow('Median Window - FG Mask', result_mask)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    print("Không đủ frame để xử lý. Hãy giảm TEST_FRAME hoặc N_WINDOW.")

[Median Window] Frame 50:
  Số object phát hiện: 3
  Toạ độ tâm object: [[26, 391], [454, 36], [417, 44]]
  Sum of intensities in fgMask: 1425960


## 3. One Last Window



In [ ]:
VIDEO_PATH = 'dataset.mp4'
TEST_FRAME = 50

capture = cv2.VideoCapture(VIDEO_PATH)

ret, prev_frame = capture.read()   # đọc frame đầu tiên làm background ban đầu
if not ret:
    print("Không mở được video!")
else:
    frame_idx = 1
    result_frame = None
    result_mask = None

    while True:
        ret, frame = capture.read()
        if not ret:
            break

        # --- Background = frame liền trước ---
        background = prev_frame

        diff = cv2.absdiff(frame, background)
        fgMask = get_foreground_mask(diff, threshold=30)

        if frame_idx == TEST_FRAME:
            result_frame = frame.copy()
            result_mask  = fgMask.copy()
            break

        prev_frame = frame   # cập nhật background
        frame_idx += 1

    capture.release()

    if result_frame is not None:
        result_display = draw_detections(result_frame, result_mask)
        locations = detect_objects(result_mask)
        print(f"[One Last Window] Frame {TEST_FRAME}:")
        print(f"  Số object phát hiện: {len(locations)}")
        print(f"  Toạ độ tâm object: {locations}")
        print(f"  Sum of intensities in fgMask: {np.sum(result_mask)}")
        cv2.imshow('One Last Window - Frame', result_display)
        cv2.imshow('One Last Window - FG Mask', result_mask)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    else:
        print("Không đủ frame để xử lý.")

[One Last Window] Frame 50:
  Số object phát hiện: 0
  Toạ độ tâm object: []
  Sum of intensities in fgMask: 187935


## 4. Mixture of Gaussians (MOG2)



In [ ]:
VIDEO_PATH = 'dataset.mp4'
TEST_FRAME = 100    # Dùng nhiều frame hơn để MOG2 học được background tốt

fgbg = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=True)

capture = cv2.VideoCapture(VIDEO_PATH)

frame_idx = 0
result_frame = None
result_mask = None

while True:
    ret, frame = capture.read()
    if not ret:
        break

    fgMask = fgbg.apply(frame)     # trả về mask: 255=foreground, 127=shadow, 0=background

    # Loại bỏ shadow (pixel = 127), giữ lại foreground (pixel = 255)
    _, fgMask = cv2.threshold(fgMask, 200, 255, cv2.THRESH_BINARY)

    # Làm sạch nhiễu
    fgMask = cv2.dilate(fgMask, None, iterations=2)
    fgMask = cv2.erode(fgMask, None, iterations=2)

    if frame_idx == TEST_FRAME:
        result_frame = frame.copy()
        result_mask  = fgMask.copy()
        break

    frame_idx += 1

capture.release()

if result_frame is not None:
    result_display = draw_detections(result_frame, result_mask)
    locations = detect_objects(result_mask)
    print(f"[MOG2] Frame {TEST_FRAME}:")
    print(f"  Số object phát hiện: {len(locations)}")
    print(f"  Toạ độ tâm object: {locations}")
    print(f"  Sum of intensities in fgMask: {np.sum(result_mask)}")
    cv2.imshow('MOG2 - Frame', result_display)
    cv2.imshow('MOG2 - FG Mask', result_mask)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    print("Không đủ frame để xử lý.")

[MOG2] Frame 100:
  Số object phát hiện: 1
  Toạ độ tâm object: [[351, 213]]
  Sum of intensities in fgMask: 748680


## So sánh 4 phương pháp

| Phương pháp | Cách tính background | Ưu điểm | Nhược điểm |
|---|---|---|---|
| **Mean Window** | Trung bình n frame | Đơn giản, ổn định | Chậm thích nghi, tốn RAM |
| **Median Window** | Trung vị n frame | Robust với outlier | Tốn tính toán hơn Mean |
| **One Last Window** | Frame trước đó | Siêu đơn giản, thích nghi nhanh | Nhạy với noise |
| **MOG2** | Hỗn hợp Gaussian | Mạnh mẽ, xử lý tốt mọi trường hợp | Tốn tài nguyên nhất |